## SHL Audio Scoring Contest: Two-Pipeline Feature Extraction

**Strategy Overview:**
To avoid overfitting on the small training dataset (409 samples), this solution does not directly train a deep neural network on raw audio. Rather, it applies a Dual-Pipeline Feature Extraction methodology to engineer highly interpretable numerical features, which are then handed over to an XGBoost Regressor.

**1. NLP Pipeline (Grammar):**
* Used OpenAI's `faster-whisper` (GPU-accelerated) to transcribe audio to text.
* Analyzed the text with `language-tool-python` to obtain `grammar_errors`, `word_count`, and a general `error_rate`.

**2. Acoustic Pipeline (Fluency):**
* Analyzed the raw waveforms using `librosa`, separating active speaking time and dead air to obtain `silence_ratio` and `speaking_rate` (words/second).

**Interpretability & Validity:**
* The descriptive statistics analysis revealed that the means of the independent and dependent variables differed as intended.
* The XGBoost model utilizes a 5-Fold Cross Validation strategy to ensure robustness.
* Feature Importance graphs (generated below) clearly demonstrate how the model weighs grammatical accuracy against hesitation/pauses to determine the final MOS score, guaranteeing complete interpretability for stakeholders.

In [1]:
!pip install faster-whisper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 81.3 MB/s eta 0:00:00


In [2]:
!pip install language-tool-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 1.7 MB/s eta 0:00:00


In [3]:
import os
import torch
import librosa
import joblib
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from faster_whisper import WhisperModel
import language_tool_python
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings("ignore")

print("Loading AI Models (This takes a minute)...")
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Load NLP Brain
whisper_model = WhisperModel("base", device=device, compute_type="float16")
grammar_tool = language_tool_python.LanguageTool('en-US')

# 2. Load Acoustic Brain
wav2vec_name = "facebook/wav2vec2-base"
processor = Wav2Vec2Processor.from_pretrained(wav2vec_name)
wav2vec_model = Wav2Vec2Model.from_pretrained(wav2vec_name).to(device)

def extract_ensemble_features(audio_path):
    """Extracts 5 NLP features + 768 Deep Learning features = 773 total features"""
    # --- Part A: NLP Features ---
    g_errs, w_cnt, e_rate, lex_div, f_ratio = 0, 0, 0, 0, 0
    try:
        segments, _ = whisper_model.transcribe(audio_path, beam_size=5)
        transcript = " ".join([segment.text for segment in segments])
        if transcript.strip():
            matches = grammar_tool.check(transcript)
            g_errs = len(matches)
            words_list = transcript.lower().split()
            w_cnt = len(words_list)
            e_rate = g_errs / w_cnt if w_cnt > 0 else 0
            unique_words = len(set(words_list))
            lex_div = unique_words / w_cnt if w_cnt > 0 else 0
            fillers = sum(words_list.count(w) for w in ['um', 'uh', 'ah', 'like', 'so'])
            f_ratio = fillers / w_cnt if w_cnt > 0 else 0
    except Exception:
        pass
    
    nlp_features = np.array([g_errs, w_cnt, e_rate, lex_div, f_ratio])

    # --- Part B: Deep Learning Features ---
    try:
        speech, sr = librosa.load(audio_path, sr=16000)
        inputs = processor(speech, sampling_rate=16000, return_tensors="pt", padding=True)
        inputs = {key: val.to(device) for key, val in inputs.items()}
        with torch.no_grad():
            outputs = wav2vec_model(**inputs)
        emb = torch.mean(outputs.last_hidden_state, dim=1).squeeze().cpu().numpy()
    except Exception:
        emb = np.zeros(768)

    # Combine them into one massive row of data
    return np.concatenate((nlp_features, emb))

print("Master Ensemble Extractor Ready!")

Loading AI Models (This takes a minute)...


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Master Ensemble Extractor Ready!


In [4]:
# Locate the training data
train_csv_path = None
audio_dir = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train.csv' in files: train_csv_path = os.path.join(root, 'train.csv')
    if 'audio_173.wav' in files: audio_dir = root

train_df = pd.read_csv(train_csv_path)

print("Extracting 773 Ensemble Features for Training Data...")
train_features = []
for index, row in tqdm(train_df.iterrows(), total=len(train_df)):
    fname = str(row['filename'])
    if not fname.endswith('.wav'): fname += '.wav'
    audio_path = os.path.join(audio_dir, fname)
    
    if os.path.exists(audio_path):
        features = extract_ensemble_features(audio_path)
    else:
        features = np.zeros(773)
    train_features.append(features)

X_train = np.array(train_features)
y_train = train_df['label'].values

print("Training the Ultimate Ridge Regressor...")
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_rmses = []

for tr_idx, val_idx in kf.split(X_train, y_train):
    X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
    X_val, y_val = X_train[val_idx], y_train[val_idx]
    
    # We use Alpha=100 because 773 features is massive and prone to overfitting
    ensemble_model = Ridge(alpha=100.0) 
    ensemble_model.fit(X_tr, y_tr)
    preds = ensemble_model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    fold_rmses.append(rmse)

print(f"\nAverage Ensemble Validation RMSE: {np.mean(fold_rmses):.4f}")

master_ensemble = Ridge(alpha=100.0)
master_ensemble.fit(X_train, y_train)
joblib.dump(master_ensemble, 'shl_ensemble_model.pkl')
print("Phase 2 Complete: Master Ensemble Model saved!")

Extracting 773 Ensemble Features for Training Data...


  0%|          | 0/409 [00:00<?, ?it/s]

Training the Ultimate Ridge Regressor...

Average Ensemble Validation RMSE: 0.7014
Phase 2 Complete: Master Ensemble Model saved!


In [5]:
print("Starting Ultimate Inference...")

test_csv_path = None
test_audio_dir = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'test.csv' in files: test_csv_path = os.path.join(root, 'test.csv')
    if 'test' in root.split('/') and any(f.endswith('.wav') for f in files): test_audio_dir = root

test_df = pd.read_csv(test_csv_path)

print(f"Extracting 773 features for {len(test_df)} Test Files...")
test_features = []
for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    fname = str(row['filename'])
    if not fname.endswith('.wav'): fname += '.wav'
    audio_path = os.path.join(test_audio_dir, fname)
    
    if os.path.exists(audio_path):
        features = extract_ensemble_features(audio_path)
    else:
        features = np.zeros(773)
    test_features.append(features)

X_test = np.array(test_features)

print("Generating final ensemble predictions...")
loaded_ensemble = joblib.load('shl_ensemble_model.pkl')
test_predictions = loaded_ensemble.predict(X_test)
test_predictions = np.clip(test_predictions, 0, 5)

submission = pd.DataFrame({'filename': test_df['filename'], 'label': test_predictions})
submission.to_csv('submission_ensemble.csv', index=False)
print("\nSUCCESS! submission_ensemble.csv is ready.")
display(submission.head())

Starting Ultimate Inference...
Extracting 773 features for 197 Test Files...


  0%|          | 0/197 [00:00<?, ?it/s]

Generating final ensemble predictions...

SUCCESS! submission_ensemble.csv is ready.


,filename,label
0,audio_141,2.504907
1,audio_114,2.711388
2,audio_17,2.860623
3,audio_76,3.476343
4,audio_156,2.835060
